Most network analyses of knowledge graphs treat the content of nodes as incidental. This project inverts that. The SEP is curated by experts, so when betweenness centrality identifies a bridge article, it is making a claim about the deep structure of philosophical inquiry itself: that certain problems are so fundamental they must be traversed to get from one area of philosophy to another. The network topology becomes a way of doing philosophy of philosophy — metacognition about the discipline encoded in data.

In [1]:
import json
import os
import pickle
import time
from urllib.parse import urljoin, urlparse

import networkx as nx
import requests
from bs4 import BeautifulSoup

In [2]:
SEP_BASE = "https://plato.stanford.edu"
SEP_HOST = "plato.stanford.edu"

session = requests.Session()
session.headers.update({
    # Set a descriptive UA with a contact; check /robots.txt before crawling.
    "User-Agent": "sep-link-grapher-research/1.0 (contact: jit.latter557@aleeas.com)"
})

In [3]:
def get_all_entries():
    """Get all SEP entry URLs from the table of contents. Returns {slug: title}."""
    r = session.get(f"{SEP_BASE}/contents.html", timeout=10)
    r.raise_for_status()
    soup = BeautifulSoup(r.text, "html.parser")
    entries = {}
    for a in soup.select("div#content a[href]"):
        href = a["href"]
        if href.startswith("entries/"):
            slug = href.rstrip("/").split("/")[-1]
            entries[slug] = a.text.strip()
    return entries

In [4]:
def get_entry_links(slug):
    """Get internal SEP entry slugs linked from one entry. Returns a set of slugs."""
    url = f"{SEP_BASE}/entries/{slug}/"
    links = set()
    try:
        r = session.get(url, timeout=10)
        r.raise_for_status()
    except requests.RequestException as e:
        print(f"  ! {slug}: {e}")
        return links

    soup = BeautifulSoup(r.text, "html.parser")
    for a in soup.select("a[href]"):
        # urljoin resolves ../ and fragments against the current page URL.
        p = urlparse(urljoin(url, a["href"]))
        # Internal entry links only: same host, path under /entries/.
        if p.netloc and p.netloc != SEP_HOST:
            continue
        if not p.path.startswith("/entries/"):
            continue
        target = p.path.split("/entries/", 1)[1].strip("/").split("/")[0]
        if target and target != slug:
            links.add(target)
    return links

In [5]:
def build_sep_graph(entries, graph_cache="sep_graph.pkl", links_cache="sep_links.json"):
    if os.path.exists(graph_cache):
        with open(graph_cache, "rb") as f:
            return pickle.load(f)

    # Per-page cache so a mid-crawl crash doesn't lose progress; reruns resume.
    if os.path.exists(links_cache):
        with open(links_cache) as f:
            crawled = {k: set(v) for k, v in json.load(f).items()}
    else:
        crawled = {}

    todo = [s for s in entries if s not in crawled]
    for i, slug in enumerate(todo):
        crawled[slug] = get_entry_links(slug)
        time.sleep(0.8)  # be polite to the SEP server
        if i % 50 == 0:
            print(f"  {i}/{len(todo)} — {entries[slug]}")
            with open(links_cache, "w") as f:
                json.dump({k: sorted(v) for k, v in crawled.items()}, f)

    with open(links_cache, "w") as f:
        json.dump({k: sorted(v) for k, v in crawled.items()}, f)

    G = nx.DiGraph()
    for slug, title in entries.items():
        G.add_node(slug, title=title)
    for slug, targets in crawled.items():
        for target in targets:
            if target in entries:
                G.add_edge(slug, target)

    with open(graph_cache, "wb") as f:
        pickle.dump(G, f)
    return G

In [6]:
if __name__ == "__main__":
    entries = get_all_entries()
    G = build_sep_graph(entries)
    print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")

  0/1861 — abduction
  50/1861 — aesthetics
  100/1861 — analogy: medieval theories of
  150/1861 — David
  200/1861 — authenticity
  250/1861 — developmental biology
  300/1861 — Cambridge Platonists
  350/1861 — Chinese and Western
  400/1861 — cognitivism vs. non-cognitivism, moral
  450/1861 — consequentialism: rule
  500/1861 — Heidegger, Martin
  550/1861 — evolution and development
  600/1861 — double effect, doctrine of
  650/1861 — epistemic paradoxes
  700/1861 — the legal concept of
  750/1861 — philosophy of language
  800/1861 — free rider problem
  850/1861 — moral arguments
  900/1861 — moral and political philosophy
  950/1861 — impartiality
  1000/1861 — inverted
  ! james-viterbo: HTTPSConnectionPool(host='plato.stanford.edu', port=443): Read timed out. (read timeout=10)
  ! japanese-philosophy: HTTPSConnectionPool(host='plato.stanford.edu', port=443): Max retries exceeded with url: /entries/japanese-philosophy/ (Caused by ConnectTimeoutError(<HTTPSConnection(host='pl

In [ ]:
import pandas as pd
import numpy as np

# Work on undirected graph for betweenness
# (standard practice: we care about conceptual proximity, not link direction)
G_u = G.to_undirected()

# Keep only the largest connected component
lcc_nodes = max(nx.connected_components(G_u), key=len)
G_lcc = G_u.subgraph(lcc_nodes).copy()
print(f"LCC: {G_lcc.number_of_nodes()} nodes, {G_lcc.number_of_edges()} edges")

# Exact betweenness — feasible at this scale
print("Computing betweenness (this takes 2–5 min)...")
bc = nx.betweenness_centrality(G_lcc, normalized=True)
cc = nx.closeness_centrality(G_lcc)
dc = nx.degree_centrality(G_lcc)
pr = nx.pagerank(G, alpha=0.85)  # use directed graph for PageRank

# Build results DataFrame with human-readable titles
df = pd.DataFrame({
    'slug':        list(bc.keys()),
    'title':       [entries.get(n, n) for n in bc.keys()],
    'betweenness': list(bc.values()),
    'closeness':   [cc[n] for n in bc],
    'degree':      [dc[n] for n in bc],
    'pagerank':    [pr.get(n, 0) for n in bc],
})

# The bridge ratio: high betweenness relative to degree
df['bridge_ratio'] = df['betweenness'] / (df['degree'] + 1e-9)

# Three ranked lists — each tells a different story
print("\nTop 15 by betweenness (structurally central):")
print(df.nlargest(15, 'betweenness')[['title','betweenness','degree']].to_string())

print("\nTop 15 by bridge ratio (hidden bridges):")
print(df.nlargest(15, 'bridge_ratio')[['title','betweenness','degree','bridge_ratio']].to_string())

print("\nTop 15 by PageRank (prestige-weighted centrality):")
print(df.nlargest(15, 'pagerank')[['title','pagerank','degree']].to_string())

In [ ]:
# Find the disagreements
top_bc  = set(df.nlargest(30, 'betweenness')['slug'])
top_pr  = set(df.nlargest(30, 'pagerank')['slug'])
top_br  = set(df.nlargest(30, 'bridge_ratio')['slug'])

# High betweenness but NOT high PageRank = underappreciated bridges
underappreciated = top_bc - top_pr
print("Structurally critical but not famous:")
print(df[df['slug'].isin(underappreciated)][['title','betweenness','pagerank']])

# High PageRank but NOT high betweenness = prestige hubs
prestige_hubs = top_pr - top_bc
print("\nFamous but not structurally critical:")
print(df[df['slug'].isin(prestige_hubs)][['title','betweenness','pagerank']])

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patheffects as pe

fig, ax = plt.subplots(figsize=(9, 6))

ax.scatter(df['degree'], df['betweenness'],
           alpha=0.35, s=18, color='#534AB7')

# Annotate top bridge-ratio entries (not just top betweenness)
to_label = df.nlargest(18, 'bridge_ratio')
for _, row in to_label.iterrows():
    ax.annotate(row['title'][:28],
                (row['degree'], row['betweenness']),
                fontsize=7.5, alpha=0.9,
                xytext=(5, 3), textcoords='offset points')

ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Degree centrality (log)')
ax.set_ylabel('Betweenness centrality (log)')
ax.set_title('Betweenness vs. degree in the SEP — top-left = hidden bridges')
plt.tight_layout()

In [ ]:
import seaborn as sns

# Manually assign subdisciplines from SEP categories
# (or use entry title keywords as a proxy)
subdiscipline_map = {
    'causation': 'Metaphysics',
    'truth': 'Philosophy of Language',
    'consciousness': 'Philosophy of Mind',
    'knowledge-analysis': 'Epistemology',
    'modality-epistemology': 'Epistemology',
    'possible-worlds': 'Metaphysics',
    # ... extend for top entries
}
df['subdiscipline'] = df['slug'].map(subdiscipline_map).fillna('Other')

avg_bc = (df.groupby('subdiscipline')['betweenness']
            .mean()
            .sort_values(ascending=False))

fig, ax = plt.subplots(figsize=(8, 4))
avg_bc.plot(kind='barh', ax=ax, color='#5DCAA5')
ax.set_xlabel('Mean betweenness centrality')
ax.set_title('Average betweenness by subdiscipline')
plt.tight_layout()

In [ ]:
from pyvis.network import Network

# Color nodes by subdiscipline
color_map = {
    'Metaphysics':             '#7F77DD',
    'Epistemology':            '#1D9E75',
    'Philosophy of Mind':      '#D85A30',
    'Philosophy of Language':  '#378ADD',
    'Ethics':                  '#EF9F27',
    'Logic':                   '#888780',
    'Other':                   '#B4B2A9',
}

# Show top 300 nodes by betweenness
top = df.nlargest(300, 'betweenness')['slug'].tolist()
G_vis = G_lcc.subgraph(top)

net = Network(height='680px', notebook=True)
for node in G_vis.nodes():
    row = df[df['slug'] == node].iloc[0]
    sub = subdiscipline_map.get(node, 'Other')
    net.add_node(node,
                 label=row['title'][:30],
                 size=row['betweenness'] * 4000 + 6,
                 color=color_map.get(sub, '#B4B2A9'),
                 title=f"{row['title']}<br>Betweenness: {row['betweenness']:.4f}<br>Bridge ratio: {row['bridge_ratio']:.2f}")
for u, v in G_vis.edges():
    net.add_edge(u, v, width=0.4, color='#D3D1C7')

net.show('sep_bridges.html')